# Phase 6: publish the online demo (Hugging Face Space)

Builds a small web app where anyone can upload a mammogram and see the model's malignancy score, density grade and
heatmaps. It uses the **paper-design model from Phase 4** (no retraining) and 6 **randomly drawn** official-test
images as examples, then uploads everything to a free Hugging Face Space.

**Before running (one-time setup, ~5 minutes)**
1. Create a free account at **huggingface.co** and confirm your e-mail.
2. On Hugging Face: your avatar → **Settings → Access Tokens → + Create new token** → token type **Write** →
   name it `kaggle` → **Create token** → copy it (starts with `hf_`).
3. In this Kaggle notebook: menu **Add-ons → Secrets → + Add Secret**. Label: `HF_TOKEN`, Value: paste the token →
   Save. Make sure the **checkbox next to HF_TOKEN is ticked** (attached to this notebook).
4. Settings → **Internet on**. Accelerator: **None (CPU) is enough**.
5. Add Input → **"CBIS-DDSM: Breast Cancer Image Dataset"** (by *awsaf49*).
6. Add Input → **Your Work → Notebooks → `notebookb86c6df37c`** (the Phase 4 notebook with `checkpoints/`).
7. Run the cells **one by one, top to bottom** (Run All is fine too). About 10 minutes in total.
8. Send Claude: the Space link printed by cell 5, and **`demo_results.zip`** from Output (cell 6).


In [ ]:
# 1) Get the code, find the Phase 4 checkpoint
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
os.environ["PYTHONPATH"] = f"{REPO}/src"
sys.path.insert(0, f"{REPO}/src")
from mammo.experiments.attention import find_checkpoints
ck = find_checkpoints("auto").get("mt_cbam")
print("paper-design checkpoint:", ck or "MISSING -> add the Phase 4 notebook (notebookb86c6df37c) as an input")


In [ ]:
# 2) Tests (includes an end-to-end build with a tiny untrained model). Must end with "passed".
!cd /kaggle/working/repo && python -m pytest -q tests/test_demo.py 2>&1 | tail -3


In [ ]:
# 3) Build the Space folder: model + code + 6 random test images; check it reproduces the Phase 5 predictions
!cd /kaggle/working/repo && python -m mammo.experiments.demo_export build
from IPython.display import Image, display
display(Image("/kaggle/working/results/demo/demo_preview.png"))


In [ ]:
# 4) Try the actual web app code once, without launching it (installs Gradio in this notebook only)
!pip install -q gradio 2>&1 | tail -1
import importlib.util
spec = importlib.util.spec_from_file_location("app", "/kaggle/working/space/app.py")
app = importlib.util.module_from_spec(spec); spec.loader.exec_module(app)
img, mal, dens, txt = app.analyse("/kaggle/working/space/examples/example_1.jpg")
print("app OK:", img.shape, mal, "\n", txt.splitlines()[0])


In [ ]:
# 5) Upload to Hugging Face (needs the HF_TOKEN secret, see the first cell). Prints the link to your demo.
!cd /kaggle/working/repo && python -m mammo.experiments.demo_export upload


In [ ]:
# 6) Pack the check files for Claude -> download demo_results.zip from Output (/kaggle/working)
!cd /kaggle/working && rm -f demo_results.zip && zip -qr demo_results.zip results/demo && ls -lh demo_results.zip
